In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os
import math
from importlib import reload

import torch
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from mars import spin_model, spectra_manager, constants, population, concat

import torch
import math
from mars.population import transform as pt  # Replace with your actual import path

import math
import torch
from typing import List


import torch


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import torch
import math
from typing import Dict, Tuple, List, Optional
from mars.population import transform as pt

# ==============================================================================
# 1. FAST KINETIC PROJECTION (Non-Secular & Secular)
# ==============================================================================
def rate_matrix_to_superoperator(W: torch.Tensor) -> torch.Tensor:
    """
    Convert a classical rate matrix W[target, source] into a Lindblad
    superoperator by summing jump operators sqrt(W[m,n]) |m><n|.
    """
    d = W.shape[-1]
    batch_shape = W.shape[:-2]
    dtype = torch.complex128 if W.dtype in (torch.float64, torch.complex128) else torch.complex64
    device = W.device

    L = torch.zeros(*batch_shape, d * d, d * d, dtype=dtype, device=device)

    for m in range(d):
        for n in range(d):
            if m == n:
                continue
            rate = W[..., m, n].clamp_min(0.0)
            if torch.all(rate == 0):
                continue

            J = torch.zeros(*batch_shape, d, d, dtype=dtype, device=device)
            J[..., m, n] = torch.sqrt(rate).to(dtype)
            L = L + pt.Liouvilleator.lindblad_dissipator_from_operator(J)

    return L



def generate_random_unitary(n: int, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
    H = torch.randn(n, n, dtype=dtype, device=device)
    Q, R = torch.linalg.qr(H)
    ph = torch.diagonal(R)
    ph = ph / ph.abs().clamp_min(1e-12)
    return Q * ph.conj()


def generate_random_rate_matrix(
    n: int,
    dtype: torch.dtype,
    device: torch.device,
) -> torch.Tensor:
    """
    Build a column-conserving classical rate matrix W[target, source].
    Off-diagonal entries are nonnegative, diagonal entries are negative
    outgoing sums.
    """
    W = torch.rand(n, n, dtype=dtype, device=device)
    W.fill_diagonal_(0.0)
    return W


def setup_test_case(
    d1: int,
    d2: int,
    dtype: torch.dtype,
    device: torch.device,
    seed: int,
    batch_size: int = 1,
) -> Dict[str, torch.Tensor]:
    torch.manual_seed(seed)
    batch_shape = (batch_size,) if batch_size > 1 else ()

    basis_1 = generate_random_unitary(d1, dtype, device)
    basis_2 = generate_random_unitary(d2, dtype, device)
    target_basis = generate_random_unitary(d1 * d2, dtype, device)

    if batch_size > 1:
        basis_1 = basis_1.unsqueeze(0).expand(batch_shape + basis_1.shape)
        basis_2 = basis_2.unsqueeze(0).expand(batch_shape + basis_2.shape)
        target_basis = target_basis.unsqueeze(0).expand(batch_shape + target_basis.shape)

    C = pt.compute_clebsch_gordan_coeffs(target_basis, [basis_1, basis_2])
    U_flat = C.reshape(*batch_shape, d1 * d2, d1 * d2)

    # Random local jump operators for test 1
    J1 = torch.randn(batch_shape + (d1, d1), dtype=dtype, device=device)
    J2 = torch.randn(batch_shape + (d2, d2), dtype=dtype, device=device)

    # Random classical rate matrices for test 2
    W1 = generate_random_rate_matrix(d1, torch.float64, device)
    W2 = generate_random_rate_matrix(d2, torch.float64, device)
    if batch_size > 1:
        W1 = W1.unsqueeze(0).expand(batch_shape + W1.shape)
        W2 = W2.unsqueeze(0).expand(batch_shape + W2.shape)

    L1_rate = pt.Liouvilleator().lindblad_dissipator_from_rates(W1).to(torch.complex128)
    L2_rate =  pt.Liouvilleator().lindblad_dissipator_from_rates(W2).to(torch.complex128)

    return {
        "basis_1": basis_1,
        "basis_2": basis_2,
        "target_basis": target_basis,
        "C": C,
        "U_flat": U_flat,
        "J1": J1,
        "J2": J2,
        "W1": W1,
        "W2": W2,
        "L1_rate": L1_rate,
        "L2_rate": L2_rate,
        "d1": d1,
        "d2": d2,
        "d": d1 * d2,
    }

# ==============================================================================
# 3. CHECK 1: SUPEROPERATOR CONSTRUCTION & BASIS TRANSFORMATION
# ==============================================================================
def check_superoperator_equivalence(case: Dict[str, torch.Tensor]) -> Dict[str, float]:
    """
    Test 1:
    Transform the jump operators first and then build the dissipator,
    versus build the dissipator first and then transform the superoperator.
    """
    J1, J2 = case["J1"], case["J2"]
    U_flat = case["U_flat"]
    d1, d2 = case["d1"], case["d2"]

    batch_shape = J1.shape[:-2] if J1.dim() > 2 else ()
    if U_flat.dim() < len(batch_shape) + 2:
        U_flat = U_flat.expand(batch_shape + U_flat.shape)

    I1 = torch.eye(d1, dtype=J1.dtype, device=J1.device).expand(batch_shape + (d1, d1))
    I2 = torch.eye(d2, dtype=J2.dtype, device=J2.device).expand(batch_shape + (d2, d2))

    J1_local = pt.batched_kron(J1, I2)
    J2_local = pt.batched_kron(I1, J2)

    # Pathway 1: transform jumps -> build superoperator
    U_dag = U_flat.conj().transpose(-1, -2)
    J1_eig = U_flat @ J1_local @ U_dag
    J2_eig = U_flat @ J2_local @ U_dag
    L_from_jumps = (
        pt.Liouvilleator.lindblad_dissipator_from_operator(J1_eig)
        + pt.Liouvilleator.lindblad_dissipator_from_operator(J2_eig)
    )

    # Pathway 2: build local superoperators -> transform total superoperator
    L1_super = pt.Liouvilleator.lindblad_dissipator_from_operator(J1)
    L2_super = pt.Liouvilleator.lindblad_dissipator_from_operator(J2)
    L_from_mars = pt.transform_kronecker_superoperator(
        [L1_super, L2_super],
        U_flat,
        apply_secular_approximation=False,
    )

    L_local = (
        pt.Liouvilleator.lindblad_dissipator_from_operator(J1_local)
        + pt.Liouvilleator.lindblad_dissipator_from_operator(J2_local)
    )
    T = pt.batched_kron(U_flat, U_flat.conj())
    L_direct = T @ L_local @ T.conj().transpose(-1, -2)

    diff_mars = torch.max(torch.abs(L_from_jumps - L_from_mars)).item()
    diff_direct = torch.max(torch.abs(L_from_jumps - L_direct)).item()

    tol = 1e-10 if J1.dtype == torch.complex128 else 1e-6
    return {
        "diff_MarS_func": diff_mars,
        "diff_Direct_ref": diff_direct,
        "MarS_pass": diff_mars < tol,
        "Direct_pass": diff_direct < tol,
        "description": "Transform jumps -> build dissipator equals build dissipator -> transform superoperator",
    }

# ==============================================================================
# 4. CHECK 2: KINETIC MATRIX EXTRACTION vs FAST PROJECTION
# ==============================================================================
def check_kinetic_equivalence(case: Dict[str, torch.Tensor], secular: bool) -> Dict[str, float]:
    """
    Test 2:
    Extract the kinetic block from the transformed superoperator and compare
    it to the direct rate-matrix transform.
    """
    W1, W2 = case["W1"], case["W2"]
    L1_rate, L2_rate = case["L1_rate"], case["L2_rate"]
    C, U_flat = case["C"], case["U_flat"]
    L_eig = pt.transform_kronecker_superoperator(
        [L1_rate, L2_rate],
        U_flat,
        apply_secular_approximation=False,
    )
    K_extracted = pt.extract_transition_matrix_from_superoperator(L_eig).real

    K_direct = pt.transform_kronecker_rate_matrix([W1, W2], pt.get_product_to_target_unitary(C, n=2))
    idx = torch.arange(K_direct.shape[-1])

    K_direct[..., idx, idx] = 0.0
    K_direct[..., idx, idx] = -K_direct.sum(dim=-2)
    abs_diff = torch.max(torch.abs(K_direct - K_extracted)).item()
    rel_diff = (torch.norm(K_direct - K_extracted) / (torch.norm(K_extracted) + 1e-12)).item()
    col_sums = torch.max(torch.abs(K_direct.sum(dim=0))).item()

    tol_abs = 1e-8
    tol_rel = 1e-6
    return {
        "abs_diff": abs_diff,
        "rel_diff": rel_diff,
        "max_col_sum": col_sums,
        "pass": abs_diff < tol_abs and rel_diff < tol_rel,
        "conservation_pass": col_sums < tol_abs,
        "description": "Kinetic block extracted from transformed superoperator equals direct rate-matrix transform",
    }

# ==============================================================================
# 5. MAIN TEST SUITE RUNNER
# ==============================================================================
def run_test_suite(num_cases: int = 6) -> None:
    """Execute multiple test cases with varying dimensions, seeds, and approximations."""
    dtype = torch.complex128
    device = torch.device("cpu")
    configs = [
        (2, 2, 0, False, 1),
        (3, 4, 42, False, 1),
        (4, 3, 123, True, 1),
        (5, 2, 99, False, 1),
        (3, 3, 777, False, 4),
        (4, 5, 2024, True, 1),
    ][:num_cases]
    
    print(f"🚀 Running {len(configs)} test configurations...")
    print("=" * 70)
    
    for i, (d1, d2, seed, secular, batch) in enumerate(configs, 1):
        print(f"\n📦 Case {i}: d1={d1}, d2={d2}, seed={seed}, secular={secular}, batch={batch}")
        print("-" * 70)
        
        case = setup_test_case(d1, d2, dtype, device, seed, batch_size=batch)

        # Check 1
        res1 = check_superoperator_equivalence(case)
        print(f"Direct_pass: {res1['Direct_pass']}")
        print(f"MarS_pass: {res1['MarS_pass']}")

        # Check 2
        res2 = check_kinetic_equivalence(case, secular=secular)
        print(f"  🔹 Kinetic Abs Diff              : {res2['abs_diff']:.2e}")
        print(f"  🔹 Kinetic Rel Diff              : {res2['rel_diff']:.2e} {'✅' if res2['pass'] else '❌'}")
        print(f"  🔹 Population Conservation (ΣK)  : {res2['max_col_sum']:.2e} {'✅' if res2['conservation_pass'] else '❌'}")

            
    print("\n" + "=" * 70)
    print("✅ Test suite complete. Verify all ✅ flags passed for production use.")
    print("=" * 70)

if __name__ == "__main__":
    run_test_suite()

🚀 Running 6 test configurations...

📦 Case 1: d1=2, d2=2, seed=0, secular=False, batch=1
----------------------------------------------------------------------
Direct_pass: True
MarS_pass: True
  🔹 Kinetic Abs Diff              : 8.88e-16
  🔹 Kinetic Rel Diff              : 5.27e-16 ✅
  🔹 Population Conservation (ΣK)  : 1.11e-16 ✅

📦 Case 2: d1=3, d2=4, seed=42, secular=False, batch=1
----------------------------------------------------------------------
Direct_pass: True
MarS_pass: True
  🔹 Kinetic Abs Diff              : 8.88e-16
  🔹 Kinetic Rel Diff              : 3.80e-16 ✅
  🔹 Population Conservation (ΣK)  : 2.22e-16 ✅

📦 Case 3: d1=4, d2=3, seed=123, secular=True, batch=1
----------------------------------------------------------------------
Direct_pass: True
MarS_pass: True
  🔹 Kinetic Abs Diff              : 3.55e-15
  🔹 Kinetic Rel Diff              : 6.59e-16 ✅
  🔹 Population Conservation (ΣK)  : 5.55e-16 ✅

📦 Case 4: d1=5, d2=2, seed=99, secular=False, batch=1
--------------